In [1]:
import geopandas as gpd
import numpy as np
from pathlib import Path

import subkart
import pandas as pd


# Snippet for small tasks

### Add additional feature layers to geoserver

In [ ]:
source = "https://storage.googleapis.com/niva-geodata/MarintNaturKart/input/kartverket/sjoekart_dybdedata_trening_norge.geo.parquet"
subkart.utils.parquet_to_postgis(source)

In [2]:
marine_vanntyper = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/input/mdir/NyTypologi2022.geo.parquet").to_crs(
    "EPSG:25833"
)
fname = "nisjedata-substrat-marine-vanntyper_norge"
marine_vanntyper = subkart.features.marine_vanntyper_preprocess(marine_vanntyper)
marine_vanntyper["beskrivelse"] = marine_vanntyper['Type'].apply(lambda x: subkart.features.MARINE_VANN_TYPE_DESC[x])
subkart.utils.to_postgis(marine_vanntyper, fname)

Table nisjedata_substrat_marine_vanntyper_norge uploaded to PostGIS.


## Create features tiffs

In [2]:
RESOLUTION = 50
nodata = 255
crs = "EPSG:25833"

In [4]:
marine_vanntyper = gpd.read_parquet("gs://niva-geodata/MarintNaturKart/input/mdir/NyTypologi2022.geo.parquet").to_crs(
    crs
)
marine_vanntyper = subkart.features.marine_vanntyper_preprocess(marine_vanntyper)
dem_norge = subkart.sources.dem_data()

In [16]:
for region_name, region_list in subkart.sources.REGIONS.items():
    print(f"Processing region: {region_name}")
    gdf_sea_map_region = subkart.sources.sea_map_basisdata(region_list)
    gdf_sea_map_region = subkart.features.depth_preprocess(gdf_sea_map_region)
    transform, out_shape, bounds = subkart.features.to_raster_shapes(gdf_sea_map_region, res=RESOLUTION)
    marine_vanntyper_region = marine_vanntyper.cx[bounds[0] : bounds[2], bounds[1] : bounds[3]]
    dem_region = subkart.utils.resample_dem(dem_norge.crop(bounds), out_shape, transform, crs)

    X, valid_attrs, out_shape, transform = subkart.features.build(
        dem_region, gdf_sea_map_region, marine_vanntyper_region, valid_mask=None, res=RESOLUTION, dtype=np.float16
    )

    subkart.utils.save_feature_rasters(region_name, X, valid_attrs, transform, out_shape, dem_region.nodata)

Processing region: sor-ost
Preparing sea_avg_depth...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_avg_slope...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_compactness...


...

Preparing sea_compactness...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing sea_convexity...


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/geoutils/raster/raster.py:1822: UserWarning: Setting default nodata -99999 to mask non-finite values found in the array, as no nodata value was defined.
  warnings.warn(


Preparing marine types...
Stacking feature arrays...


In [ ]:
for name in subkart.features.DEPTH_NAMES:
    file_list = [Path(f"features/{region_name}_{name}.tif") for region_name in subkart.sources.REGIONS.keys()]

    subkart.utils.merge_rasters(file_list, Path(f"features/{name}_merged.tif"), nodata=dem_region.nodata)

## Create Preprocessed Depth data

In [2]:
county_files = []
for f in subkart.sources.FYLKER:
    code = next((c for c in subkart.sources.FYLKER if c.endswith(f)))
    county_files.append(
        Path(f"../geonorge/Basisdata_{code}_25833_Dybdedata_Dybdeareal.geo.parquet")
    )

gdfs = [gpd.read_parquet(path) for path in county_files]

gdf_depth = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

gdf = subkart.features.depth_preprocess(gdf_depth, is_rerun=True)

gdf.to_parquet(Path("../geonorge/sjoekart_dybdedata_trening_norge.geo.parquet"))

## Simplified AOI from marine vanntyper dataset

In [2]:
subkart.sources.prepare_mv_aoi()

In [2]:
subkart.sources.prepare_land_data()

Written GeoPackage to /home/kim/work/marint-naturkart-nivaR/geonorge/Basisdata_Landareal.gpkg
